In [ ]:
from pyspark.sql import functions as F, Window

cities = spark.read.option("header", True).option("inferSchema", True).csv("Files/config/cities.csv")

dim_city = (cities
    .withColumn("city_key", F.row_number().over(Window.orderBy("city")))
    .select("city_key", "city", "region",
            F.col("lat").alias("latitude"), F.col("lon").alias("longitude")))

dim_city.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_city")
dim_city.orderBy("city_key").show(truncate=False)

In [ ]:
bounds = (spark.table("gold_weather_daily").select("date")
          .unionByName(spark.table("gold_air_quality_daily").select("date"))
          .agg(F.min("date").alias("d0"), F.max("date").alias("d1")).first())

dim_date = (spark.range(1).select(
        F.explode(F.sequence(F.lit(bounds.d0), F.lit(bounds.d1), F.expr("interval 1 day"))).alias("date"))
    .select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        "date",
        F.year("date").alias("annee"),
        F.month("date").alias("mois"),
        F.date_format("date", "MMMM").alias("nom_mois"),
        F.dayofmonth("date").alias("jour"),
        F.date_format("date", "EEEE").alias("nom_jour"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend")))

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dim_date")
print("dim_date :", dim_date.count(), "jours,", bounds.d0, "->", bounds.d1)

In [ ]:
def build_fact(gold_name, fact_name):
    dc = spark.table("dim_city").select("city_key", "city")
    f = (spark.table(gold_name)
         .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
         .join(dc, "city", "left")
         .drop("city", "date"))
    cols = ["city_key", "date_key"] + [c for c in f.columns if c not in ("city_key", "date_key")]
    f.select(*cols).write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(fact_name)

build_fact("gold_weather_daily", "fact_weather_daily")
build_fact("gold_air_quality_daily", "fact_air_quality_daily")

In [ ]:
for t in ["dim_city", "dim_date", "fact_weather_daily", "fact_air_quality_daily"]:
    print(t, spark.table(t).count())

for t in ["fact_weather_daily", "fact_air_quality_daily"]:
    orph_city = spark.table(t).filter("city_key is null").count()
    orph_date = (spark.table(t).join(spark.table("dim_date"), "date_key", "left_anti").count())
    dup = spark.table(t).groupBy("city_key", "date_key").count().filter("count > 1").count()
    print(f"{t}: villes sans clé={orph_city}, dates sans clé={orph_date}, doublons grain={dup}")